## Setup

In [1]:
import warnings
import langchain, langchain_core  # noqa: F401

# Silence three known, harmless library warnings that would otherwise flood the output.
# The langchain and langchain_core packages re-enable their own warnings on import, so they come first.
warnings.filterwarnings("ignore", message="Relevance scores must be between 0 and 1")
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy import create_engine
import utils
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.store.sqlite import SqliteStore
import json

In [2]:
load_dotenv()

True

## Run

In [3]:
solution_dir = Path(utils.__file__).resolve().parent

cultpass_db = solution_dir / "data" / "external" / "cultpass.db"
udahub_db = solution_dir / "data" / "core" / "udahub.db"

# Check before connecting: SQLite could otherwise create an empty database.
for database_path in (cultpass_db, udahub_db):
    # `is_file` checks that an existing path points to a file.
    if not database_path.is_file():
        raise FileNotFoundError(
            f"Missing database: {database_path}. "
            "Run setup notebooks 01 and 02 first."
        )

# `create_engine` prepares database access; connections open when used.
cultpass_engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
udahub_engine = create_engine(f"sqlite:///{udahub_db}", echo=False)

print("Both database engines are ready.")

Both database engines are ready.


In [4]:
# Create persistent conversation checkpoint
memory_dir = solution_dir / "data" / "memory"

# `mkdir` creates missing parent directories and accepts an existing folder.
memory_dir.mkdir(parents=True, exist_ok=True)

checkpoint_db = memory_dir / "checkpoints.sqlite"

# `check_same_thread=False` permits LangGraph worker threads to use this connection.
checkpoint_connection = sqlite3.connect(
    str(checkpoint_db),
    check_same_thread=False,
)

# SqliteSaver saves graph state and conversation checkpoints in SQLite.
checkpointer = SqliteSaver(checkpoint_connection)

# `setup` creates the checkpoint tables when they do not already exist.
checkpointer.setup()

print("Conversation checkpointing is ready.")

Conversation checkpointing is ready.


In [5]:
from agentic.workflow import build_workflow, AgentModels, RetrievalSettings
from agentic.workflow import WorkflowLimits
from uuid import uuid4

from langchain_core.documents import Document
from langchain_chroma import Chroma
from data.models import udahub

limits: WorkflowLimits = {
    "max_graph_steps": 20,
    "max_model_calls": 15,
    "call_timeout_seconds": 30.0,
}

chat_model_name = "gpt-4o-mini"
embedding_model_name = "text-embedding-3-small"

if not chat_model_name or not embedding_model_name:
    raise ValueError("Set both model names before running this cell.")

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

if not api_key:
     api_key=os.getenv("VOCAREUM_API_KEY")
     if api_key and not base_url:
          raise ValueError("Set OPENAI_BASE_URL to your Vocareum API endpoint.")

if not api_key:
    raise ValueError("Set OPENAI_API_KEY or VOCAREUM_API_KEY in your .env file.")

chat_model = ChatOpenAI(
     model_name=chat_model_name, 
     # Temperature 0 keeps classification and structured outputs consistent.
     temperature=0,
     api_key=api_key, 
     base_url=base_url,
     timeout = limits["call_timeout_seconds"],
     max_retries = 0,
     )
embeddings = OpenAIEmbeddings(
     model=embedding_model_name,
     api_key=api_key,
     base_url=base_url,
     timeout = limits["call_timeout_seconds"],
     max_retries = 0,
)

print("Chat and embedding clients are ready.")

# Create the persistent long term memory store.

memory_db = memory_dir / "customer_memory.sqlite"

embedding_dimensions = 1536

memory_connection = sqlite3.connect(
     str(memory_db),
     check_same_thread=False,
     isolation_level=None,
)

store = SqliteStore(
     memory_connection,
     index ={
          "dims": embedding_dimensions,
          "embed": embeddings,
          "fields": ["content"],
     },
)

store.setup()

print ("Persistent long-term memory store is ready.")

# Build the cultpass article index

knowledge_account_id = "cultpass"

with utils.get_session(udahub_engine) as session:
     articles = (
          session.query(udahub.Knowledge)
          .filter(udahub.Knowledge.account_id == knowledge_account_id)
          .all()
     )

     documents= [
          Document(
               page_content=article.content,
               metadata = {
                    "article_id": article.article_id,
                    "account_id": article.account_id,
                    "title": article.title
               },
          )
          for article in articles
     ]
if not documents:
     raise ValueError("No CultPass articles were found. Complete notebook 2 first.")

collection_name = f"cultpass_articles_{uuid4().hex}"

knowledge_store = Chroma.from_documents(
     documents = documents,
     embedding = embeddings,
     ids = [document.metadata["article_id"] for document in documents],
     collection_name = collection_name,
)

print (f"Indexed {len(documents)} CultPass articles.")

# Inspect retrieval scores before selecting the gate
retrieval_test_queries = {
    "supported_login":
        "I requested a password reset but the email has not arrived.",

    "supported_reservation":
        "How can I check the status of my existing reservation?",

    "supported_availability":
        "Are there places available for the museum experience?",

    "supported_blocked_account":
        "My CultPass account is blocked. Can you restore access?",

    "supported_refund_guidance":
        "I want a refund for an unexpected CultPass charge.",

    "related_but_missing_financial_facts":
        "What exact amount will CultPass charge my bank account next month?",

    "unsupported_passport":
        "How do I renew my passport?",
}

if any(not query.strip() for query in retrieval_test_queries.values()):
     raise ValueError("Fill both retrieval test questions before running.")

retrieval_observations = []

for test_case, query in retrieval_test_queries.items():
    # This returns Document/score pairs matching the metadata filter.
    # Inspect every indexed article here; runtime candidate count comes later.
    candidates = knowledge_store.similarity_search_with_relevance_scores(
        query,
        k=len(documents),
        filter={"account_id": knowledge_account_id},
    )

    print(f"\n{test_case.upper()}: {query}")

    for document, score in candidates:
        retrieval_observations.append({
            "test_case": test_case,
            "query": query,
            "article_id": document.metadata["article_id"],
            "title": document.metadata["title"],
            "content": document.page_content,
            "score": float(score),
        })

        print(f"{score:.4f} | {document.metadata['title']}")



Chat and embedding clients are ready.
Persistent long-term memory store is ready.
Indexed 14 CultPass articles.

SUPPORTED_LOGIN: I requested a password reset but the email has not arrived.
0.2979 | Password Reset Email Has Not Arrived
0.1950 | How to Handle Login Issues?
-0.0509 | Refund Requests and Unexplained Charges
-0.1181 | Cancellation and Pause Requests: What Support Can Confirm
-0.1403 | Support for a Blocked CultPass Account
-0.1420 | Finding and Checking an Existing Reservation
-0.1473 | How to Reserve a Spot for an Event
-0.1979 | Checking the Current Subscription Status
-0.2031 | Checking Available Places Before Booking
-0.2099 | Checking an Experience's Date and Location
-0.2391 | How to Cancel or Pause a Subscription
-0.2474 | Premium Experiences and Additional Charges
-0.2479 | Monthly Allowance and Remaining Experiences
-0.2668 | What's Included in a CultPass Subscription

SUPPORTED_RESERVATION: How can I check the status of my existing reservation?
0.2557 | Finding a

In [6]:
# Retrieval settings, fixed after reading the score probe above.
# Fixed values keep the retrieval gate the same on every run; a model-chosen threshold
# varied between runs on identical scores. Recalibrate with the probe if the
# articles, embedding model or vector-store distance setting change.
calibration_conditions = {
    "embedding_model": "text-embedding-3-small",
    "article_count": 14,
}
retrieval_threshold = 0.15
retrieval_candidate_count = 3

current_conditions = {
    "embedding_model": embedding_model_name,
    "article_count": len(documents),
}
if current_conditions != calibration_conditions:
    print(
        "WARNING: retrieval settings were calibrated for",
        calibration_conditions,
        "but the current setup is",
        current_conditions,
        "- rerun the score probe and record new values.",
    )

retrieval_settings: RetrievalSettings = {
    "threshold": retrieval_threshold,
    "candidate_count": retrieval_candidate_count,
    "embedding_model": embedding_model_name,
    "score_convention": "higher_is_more_relevant",
}

print("Retrieval settings are ready:", retrieval_settings)


Retrieval settings are ready: {'threshold': 0.15, 'candidate_count': 3, 'embedding_model': 'text-embedding-3-small', 'score_convention': 'higher_is_more_relevant'}


In [7]:
from math import isfinite

calibration_rows = []

for test_case in retrieval_test_queries:
    rows = [
        row for row in retrieval_observations
        if row["test_case"] == test_case
    ][:retrieval_settings["candidate_count"]]

    for row in rows:
        score = row["score"]

        # `isfinite` rejects infinity and NaN.
        valid_score = isfinite(score) and 0.0 <= score <= 1.0

        calibration_rows.append({
            **row,
            "threshold": retrieval_settings["threshold"],
            "score_valid": valid_score,
            "gate_pass": (
                valid_score
                and score >= retrieval_settings["threshold"]
            ),
        })

calibration_report = {
    "settings": dict(retrieval_settings),
    "calibration_conditions": calibration_conditions,
    "results": calibration_rows,
}

print(json.dumps(calibration_report, indent=2, ensure_ascii=False))

{
  "settings": {
    "threshold": 0.15,
    "candidate_count": 3,
    "embedding_model": "text-embedding-3-small",
    "score_convention": "higher_is_more_relevant"
  },
  "calibration_conditions": {
    "embedding_model": "text-embedding-3-small",
    "article_count": 14
  },
  "results": [
    {
      "test_case": "supported_login",
      "query": "I requested a password reset but the email has not arrived.",
      "article_id": "c4bc5cba-b503-4000-89ef-2cfb50508eb9",
      "title": "Password Reset Email Has Not Arrived",
      "content": "Ask the customer to use 'Forgot Password' on the CultPass login screen and check that they entered their registered email. Suggest checking the spam folder. If the reset message has not arrived, retry after 10 minutes, as described in the login guidance. Persistent reset-email or login problems require human support. Do not claim that support sent an email or confirmed its delivery.",
      "score": 0.2979118126530387,
      "threshold": 0.15,
   

In [8]:
# Bind the models and compile the workflow

models: AgentModels = {
    "supervisor": chat_model,
    "classifier": chat_model,
    "knowledge": chat_model,
    "operations": chat_model,
    "response": chat_model,
    "escalation": chat_model
}

orchestrator = build_workflow(
    cultpass_engine = cultpass_engine,
    udahub_engine = udahub_engine,
    models=models,
    knowledge_store = knowledge_store,
    retrieval_settings=retrieval_settings,
    limits=limits,
    checkpointer=checkpointer,
    store=store,
)

print("Workflow compiled successfully.")

# Graph structure and diagram text.
print(orchestrator.get_graph().draw_mermaid())
demo_runs = []

Workflow compiled successfully.
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	supervisor(supervisor)
	classifier(classifier)
	knowledge(knowledge)
	operations(operations)
	response(response)
	escalation(escalation)
	persist(persist)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	classifier --> supervisor;
	escalation --> supervisor;
	intake --> supervisor;
	knowledge --> supervisor;
	operations --> supervisor;
	response --> supervisor;
	supervisor -.-> classifier;
	supervisor -.-> escalation;
	supervisor -.-> knowledge;
	supervisor -.-> operations;
	supervisor -.-> persist;
	supervisor -.-> response;
	persist --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [9]:
# Process pending database messages and display each saved result.
demo_runs = utils.batch_interface(
    orchestrator,
    udahub_engine,
    account_id=knowledge_account_id,
)

if not demo_runs:
    print("No pending customer messages on open tickets.")



Ticket: a7b865ea-87b7-4f55-9eff-01f3604a81f1
Outcome: resolved
Routing: email -> intake -> supervisor -> classifier -> supervisor -> knowledge -> supervisor -> response -> supervisor -> persist -> customer (answer)
Subject: Password Reset Instructions

Hello Bob,

Welcome back! To reset your password, please tap 'Forgot Password' on the login screen and ensure you enter your registered email. If you don't receive the reset email, check your spam folder or try again in a few minutes. If issues persist, please reach out for further assistance.

Best regards,

Ticket: 7e031a8b-c77f-4c24-b2ff-b38335cfcf3d
Outcome: resolved
Routing: email -> intake -> supervisor -> classifier -> supervisor -> knowledge -> supervisor -> response -> supervisor -> persist -> customer (answer)
Subject: Password Reset Steps

Hello Bob,

Welcome back! Here are the steps to reset your password: Please tap 'Forgot Password' on the login screen and ensure you enter your registered email. If you don't receive the re

In [10]:
# Inspect every attempted ticket after automatic processing, before cleanup.
from agentic.tools.memory_tools import search_customer_memory

demo_inspections = []
seen_tickets = set()
for run in demo_runs:
    inspected_ticket_id = run["ticket_id"]
    if inspected_ticket_id in seen_tickets:
        continue
    seen_tickets.add(inspected_ticket_id)
    inspection_config = {"configurable": {"thread_id": inspected_ticket_id}}
    try:
        # `get_state_history` yields checkpoints scoped to this thread.
        history = list(orchestrator.get_state_history(config=inspection_config))
        # `get_state` returns the latest saved values and unfinished nodes.
        snapshot = orchestrator.get_state(inspection_config)
        latest = snapshot.values
        stored_memory = (
            search_customer_memory(latest["account_id"], latest["customer_id"], store=store)
            if latest.get("account_id") and latest.get("customer_id") else []
        )
        demo_inspections.append({
            "ticket_id": inspected_ticket_id,
            "checkpoint_count": len(history),
            "pending_nodes": list(snapshot.next),
            "outcome": latest.get("outcome"),
            "messages": [{"id": msg.id, "role": msg.type, "content": msg.content}
                         for msg in latest.get("messages", [])],
            "tool_usage": latest.get("tool_results", []),
            "trace": latest.get("trace", []),
            "stored_memory": stored_memory,
            # The route each saved turn took, built from its routing_decision events.
            "routing": [utils.routing_path(turn["trace"], channel=turn.get("channel"),
                                           outcome=turn.get("outcome"),
                                           reply_present=turn.get("reply") is not None)
                        for turn in demo_runs
                        if turn["ticket_id"] == inspected_ticket_id and "trace" in turn],
            "turns": [item for item in demo_runs if item["ticket_id"] == inspected_ticket_id],
        })
    except Exception as exc:
        demo_inspections.append({"ticket_id": inspected_ticket_id,
                                 "inspection_error": type(exc).__name__})

# These are operator inspection records, including approved conversation history.
print(json.dumps(demo_inspections, indent=2, ensure_ascii=False))


[
  {
    "ticket_id": "a7b865ea-87b7-4f55-9eff-01f3604a81f1",
    "checkpoint_count": 11,
    "pending_nodes": [],
    "outcome": "resolved",
    "messages": [
      {
        "id": "c68c341c-293b-433b-9331-815e1c051d7e",
        "role": "human",
        "content": "How do I reset my password? Please keep replies concise."
      },
      {
        "id": "c68c341c-293b-433b-9331-815e1c051d7e:reply",
        "role": "ai",
        "content": "Subject: Password Reset Instructions\n\nHello Bob,\n\nWelcome back! To reset your password, please tap 'Forgot Password' on the login screen and ensure you enter your registered email. If you don't receive the reset email, check your spam folder or try again in a few minutes. If issues persist, please reach out for further assistance.\n\nBest regards,"
      }
    ],
    "tool_usage": [],
    "trace": [
      {
        "event_id": "c68c341c-293b-433b-9331-815e1c051d7e:intake:preparation",
        "turn_id": "c68c341c-293b-433b-9331-815e1c051d7e",
  

In [11]:
# Close database resources after processing and inspection.


checkpoint_connection.close()
memory_connection.close()


cultpass_engine.dispose()
udahub_engine.dispose()

print("Database resources closed.")

Database resources closed.
